# C++ OOP Fundamentals


Topics:
1. Classes and objects
2. Constructors and destructors
3. Access control (public/private/protected)
4. Member functions and `const` correctness
5. Operator overloading
6. Inheritance and polymorphism
7. Virtual functions

---

In [1]:
#include <iostream>
#include <string>
#include <cmath>
using namespace std;

## 1. Classes and Objects

In [2]:
class BankAccount {
private:
    string owner;
    double balance;
    int transactions;

public:
    // Constructor
    BankAccount(const string& name, double initialBalance = 0.0)
        : owner(name), balance(initialBalance), transactions(0) {
        cout << "Account created for " << owner << endl;
    }

    // Destructor
    ~BankAccount() {
        cout << "Account closed for " << owner << endl;
    }

    void deposit(double amount) {
        if (amount > 0) { balance += amount; transactions++; }
    }

    bool withdraw(double amount) {
        if (amount > balance) return false;
        balance -= amount;
        transactions++;
        return true;
    }

    // const member function — does not modify object
    void printStatement() const {
        cout << "Owner: " << owner
             << "  Balance: $" << balance
             << "  Transactions: " << transactions << endl;
    }

    double getBalance() const { return balance; }
};

In [3]:
{
    BankAccount acc("James", 500.0);
    acc.deposit(200.0);
    acc.deposit(150.0);
    bool ok = acc.withdraw(100.0);
    cout << "Withdrawal " << (ok ? "succeeded" : "failed") << endl;
    acc.printStatement();
}  // Destructor called here

Account created for James
Withdrawal succeeded
Owner: James  Balance: $750  Transactions: 3
Account closed for James


## 2. Operator Overloading

In [4]:
class Vec2D {
public:
    double x, y;
    Vec2D(double x = 0, double y = 0) : x(x), y(y) {}

    Vec2D operator+(const Vec2D& o) const { return {x + o.x, y + o.y}; }
    Vec2D operator-(const Vec2D& o) const { return {x - o.x, y - o.y}; }
    Vec2D operator*(double scalar) const { return {x * scalar, y * scalar}; }
    double dot(const Vec2D& o) const { return x * o.x + y * o.y; }
    double length() const { return sqrt(x*x + y*y); }

    friend ostream& operator<<(ostream& os, const Vec2D& v) {
        return os << "(" << v.x << ", " << v.y << ")";
    }
};

Vec2D a(3, 4), b(1, 2);
cout << "a = " << a << "  b = " << b << endl;
cout << "a + b = " << (a + b) << endl;
cout << "a - b = " << (a - b) << endl;
cout << "a * 2 = " << (a * 2) << endl;
cout << "a.dot(b) = " << a.dot(b) << endl;
cout << "|a| = " << a.length() << endl;

a = (3, 4)  b = (1, 2)
a + b = (4, 6)
a - b = (2, 2)
a * 2 = (6, 8)
a.dot(b) = 11
|a| = 5


## 3. Inheritance and Polymorphism

In [5]:
class Shape {
protected:
    string color;
public:
    Shape(const string& c) : color(c) {}
    virtual ~Shape() = default;

    // Pure virtual — must be implemented by derived classes
    virtual double area() const = 0;
    virtual double perimeter() const = 0;
    virtual string type() const = 0;

    void describe() const {
        cout << type() << " (" << color << "):"
             << "  area=" << area()
             << "  perimeter=" << perimeter() << endl;
    }
};

In [6]:
class Circle : public Shape {
    double radius;
public:
    Circle(double r, const string& c = "red") : Shape(c), radius(r) {}
    double area()      const override { return M_PI * radius * radius; }
    double perimeter() const override { return 2 * M_PI * radius; }
    string type()      const override { return "Circle"; }
};

class Rectangle : public Shape {
    double w, h;
public:
    Rectangle(double w, double h, const string& c = "blue") : Shape(c), w(w), h(h) {}
    double area()      const override { return w * h; }
    double perimeter() const override { return 2 * (w + h); }
    string type()      const override { return "Rectangle"; }
};

class Triangle : public Shape {
    double a, b, c;
public:
    Triangle(double a, double b, double c, const string& col = "green")
        : Shape(col), a(a), b(b), c(c) {}
    double perimeter() const override { return a + b + c; }
    double area() const override {
        double s = perimeter() / 2;
        return sqrt(s*(s-a)*(s-b)*(s-c));  // Heron's formula
    }
    string type() const override { return "Triangle"; }
};

In [7]:
#include <vector>
#include <memory>

// Polymorphism via base class pointers
vector<unique_ptr<Shape>> shapes;
shapes.push_back(make_unique<Circle>(5.0));
shapes.push_back(make_unique<Rectangle>(4.0, 6.0));
shapes.push_back(make_unique<Triangle>(3.0, 4.0, 5.0));

for (const auto& s : shapes) {
    s->describe();
}

Circle (red):  area=78.5398  perimeter=31.4159
Rectangle (blue):  area=24  perimeter=20
Triangle (green):  area=6  perimeter=12
